# Notebook 10 — p-Values and Misinterpretations

**Module:** 03 — Statistics and Probability  
**Notebook:** 10 of 18  
**Prerequisites:** NB05  
**Time estimate:** 90 minutes

> **Track A note:** Being able to state the correct definition of a p-value and
> name three common misinterpretations is a direct interview question.

---
## Step 1 — Motivation

Ioannidis (2005) argued — with mathematics — that most published findings based on
p-values are false. This was not hyperbole. The replication crisis in psychology,
medicine, and parts of biology is a direct consequence of systematic p-value
misinterpretation, publication bias, and underpowered studies. This notebook makes
you immune to the most common errors.

---
## Step 2 — Intuition

The p-value answers one very specific question: 'If the null hypothesis were true,
how often would I see data at least this extreme?' It does not answer:
- 'What is the probability H₀ is true?'
- 'How big is the effect?'
- 'Will this replicate?'
- 'Is this biologically important?'

p < 0.05 was originally proposed as a rule of thumb by Fisher — he never intended
it as a binary discovery threshold.

---
## Step 3 — Biological Background

**Why 'most published findings are false' (Ioannidis 2005):**

Given: prior probability a tested hypothesis is true = $R/(R+1)$ where $R$ is the
pre-study odds. For exploratory genomics studies, $R$ is small (most hypotheses are
false). With α=0.05, power=80%, and small $R$:

$$\text{PPV} = \frac{(1-\beta) R}{(1-\beta) R + \alpha} \ll 1$$

This is Bayes' theorem applied to hypothesis testing — the same calculation as NB03's
diagnostic test, but with hypotheses instead of patients.

**Publication bias:** negative results are not published. This inflates the apparent
discovery rate — the published literature shows only the top of a distribution of
effect size estimates, biased upward.

**Effect size matters:** a gene with p=0.001 but fold change=1.02 is statistically
significant but biologically irrelevant (with large enough n). Always report both.

---
## Step 4 — Mathematical Explanation

**The five most common p-value misinterpretations — refuted:**

| Misinterpretation | Why it's wrong |
|------------------|----------------|
| p = probability $H_0$ is true | p is computed *assuming* $H_0$ is true — it cannot tell you whether $H_0$ is true |
| 1-p = probability $H_1$ is true | p is not $P(H_0)$; $1-p$ is not $P(H_1)$ |
| p = probability of a false positive | p = $P(\text{data} | H_0)$, not $P(H_0 | \text{data})$ (prosecutor's fallacy) |
| p < 0.05 = important result | p depends on n; with large n, trivial effects become 'significant' |
| p > 0.05 = no effect | Failure to reject ≠ evidence of absence (may be underpowered) |

**Positive Predictive Value of a significant p-value:**
$$\text{PPV} = \frac{(1-\beta) \cdot R}{(1-\beta) \cdot R + \alpha \cdot 1}$$
where $R$ = pre-study odds in favour of $H_1$, $1-\beta$ = power, $\alpha$ = significance threshold.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — PPV of a significant p-value (Ioannidis framework)
def ppv_hypothesis(R: float, power: float, alpha: float = 0.05) -> float:
    """
    Positive Predictive Value of a significant result.

    Parameters
    ----------
    R     : pre-study odds H1:H0 (e.g. 0.1 = 1 in 10 hypotheses are true)
    power : 1 - beta (probability of detecting a true effect)
    alpha : significance threshold
    """
    return (power * R) / (power * R + alpha)

scenarios = [
    ("GWAS (1 true in 1000)",    0.001, 0.80),
    ("Candidate gene study",      0.10,  0.80),
    ("Well-founded hypothesis",   1.00,  0.80),
    ("Underpowered (20%)",        0.10,  0.20),
    ("GWAS + Bonferroni",         0.001, 0.80),   # alpha=5e-8 for GWAS
]

print(f"{'Scenario':<35} {'R':>6} {'Power':>7} {'α':>10} {'PPV':>8}")
print("-" * 72)
for label, R, pwr in scenarios:
    alpha_use = 5e-8 if "Bonferroni" in label else 0.05
    ppv = ppv_hypothesis(R, pwr, alpha_use)
    print(f"  {label:<33} {R:>6.3f} {pwr:>7.2f} {alpha_use:>10.2e} {ppv:>8.3f}")

In [ ]:
# Cell 6.2 — Effect size vs. p-value: large n makes trivial effects 'significant'
rng = np.random.default_rng(42)
true_effect = 0.01  # tiny biological effect (1% difference in expression)

print(f"True effect size (Cohen's d ≈ 0.01): biologically trivial")
print(f"{'n per group':>14} {'p-value':>12} {'Conclusion'}")
print("-" * 50)
for n in [10, 100, 1000, 10000, 100000]:
    x = rng.normal(0, 1, n)
    y = rng.normal(true_effect, 1, n)
    _, p = stats.ttest_ind(x, y)
    conclusion = "p < 0.05 (SIGNIFICANT)" if p < 0.05 else "ns"
    print(f"  {n:>12,}  {p:>12.4e}  {conclusion}")

print("\nLesson: statistical significance ≠ biological importance")

In [ ]:
# Cell 6.3 — P-value under H0 is uniform — demonstration
rng2 = np.random.default_rng(0)
n_per_group, n_sim = 30, 10_000

null_pvals = np.array([
    stats.ttest_ind(rng2.normal(0,1,n_per_group), rng2.normal(0,1,n_per_group))[1]
    for _ in range(n_sim)
])

# Under H0, p-values should be uniform on [0,1]
ks_stat, ks_p = stats.kstest(null_pvals, 'uniform')
print(f"KS test vs. Uniform(0,1): stat={ks_stat:.4f}, p={ks_p:.4f}")
print(f"p-values < 0.05: {(null_pvals < 0.05).mean():.3f} (expected 0.05)")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — PPV heatmap + p-value histogram under H0
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: PPV as a function of R and power
ax = axes[0]
R_vals = np.logspace(-3, 1, 100)
power_vals = [0.20, 0.50, 0.80, 0.95]
colors_ppv = ["tomato", "orange", "steelblue", "green"]
for power, color in zip(power_vals, colors_ppv):
    ppvs = [ppv_hypothesis(R, power) for R in R_vals]
    ax.semilogx(R_vals, ppvs, lw=2, color=color, label=f"Power={power:.0%}")
ax.axhline(0.5, color="black", ls="--", lw=1, label="PPV = 0.5 (50% false positives)")
ax.set_xlabel("Pre-study odds R (log scale)"); ax.set_ylabel("PPV")
ax.set_title("PPV of significant result (Ioannidis framework)")
ax.legend(fontsize=8)

# Panel 2: null p-value histogram
ax = axes[1]
ax.hist(null_pvals, bins=40, color="steelblue", edgecolor="none", density=True)
ax.axhline(1.0, color="tomato", ls="--", lw=1.5, label="Uniform(0,1) density")
ax.set_xlabel("p-value"); ax.set_ylabel("Density")
ax.set_title("p-values under H₀ are Uniform — proving the definition")
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Read Ioannidis (2005). In your own words (one paragraph): what is the key argument?
   Why does high α AND low prior probability AND low power combine badly?
2. In Cell 6.2: compute Cohen's d for each sample size scenario. Is it the same across
   all n? What does this tell you about the relationship between effect size and p-value?
3. Implement `ppv_hypothesis()` and compute it for a scenario of your choice.
   What prior odds R would be needed for PPV > 0.9 with α=0.05 and power=0.8?
4. What is the 'file drawer problem'? How does it interact with the multiple testing problem
   in genomics? Write three sentences.

---
## Quiz — Active Recall (Track A critical)

1. State the correct definition of a p-value.
2. Name three common misinterpretations of p-values.
3. Why does a p-value depend on sample size? Give an example.
4. What is the difference between statistical significance and biological significance?
5. What is publication bias? How does it affect the literature?

---
## Papers Referenced

- Ioannidis (2005), *Why Most Published Research Findings Are False*. DOI: 10.1371/journal.pmed.0020124

---
## Reflection

**Date completed:** ____________________

> *[Could you refute each of the five misinterpretations without notes? Can you explain Ioannidis's argument in 60 seconds?]*

---
*Next: `11_multiple_testing_correction.ipynb`*